In [1]:
import os
import re
import json
import glob
import hashlib
import pandas as pd

# =========================================================
# INPUT FILES
# =========================================================

OWNERSHIP_JSON_PATTERN = "gemini-code-*.json"
COMPANY_MASTER_CSV = "cse_company_master_with_ids.csv"

OUTPUT_DIR = "cse_graph"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NODES_CSV = f"{OUTPUT_DIR}/nodes.csv"
EDGES_CSV = f"{OUTPUT_DIR}/edges.csv"
ALIASES_CSV = f"{OUTPUT_DIR}/entity_aliases.csv"
REVIEW_CSV = f"{OUTPUT_DIR}/edges_manual_review.csv"

MIN_CONFIDENCE = 0.75

# =========================================================
# MANUAL ALIAS MAP
# Add more as you discover duplicates
# =========================================================

MANUAL_ALIASES = {
    # People
    "K D D PERERA": "DHAMMIKA PERERA",
    "K. D. D. PERERA": "DHAMMIKA PERERA",
    "MR K D D PERERA": "DHAMMIKA PERERA",
    "MR. K. D. D. PERERA": "DHAMMIKA PERERA",
    "KULAPPUGE DON DHAMMIKA PERERA": "DHAMMIKA PERERA",

    "A K PATHIRAGE": "ASHOK PATHIRAGE",
    "A. K. PATHIRAGE": "ASHOK PATHIRAGE",
    "MR A K PATHIRAGE": "ASHOK PATHIRAGE",
    "MR. A. K. PATHIRAGE": "ASHOK PATHIRAGE",

    "S J S PERERA": "SUMAL PERERA",
    "S. J. S. PERERA": "SUMAL PERERA",

    # Government / institutions
    "GOSL": "GOVERNMENT OF SRI LANKA",
    "GOVERNMENT OF SRI LANKA": "GOVERNMENT OF SRI LANKA",
    "URBAN DEVELOPMENT AUTHORITY": "URBAN DEVELOPMENT AUTHORITY",
    "URBAN DEVELOPMENT AUTHORITY UDA": "URBAN DEVELOPMENT AUTHORITY",

    # Companies common variants
    "E B CREASY & COMPANY PLC": "E B CREASY AND COMPANY PLC",
    "E. B. CREASY & COMPANY PLC": "E B CREASY AND COMPANY PLC",
    "E B CREASY AND COMPANY PLC": "E B CREASY AND COMPANY PLC",

    "BROWN & COMPANY PLC": "BROWN AND COMPANY PLC",
    "BROWN AND COMPANY PLC": "BROWN AND COMPANY PLC",

    "SINGER SRI LANKA PLC": "SINGER (SRI LANKA) PLC",
    "SINGER (SRI LANKA) PLC": "SINGER (SRI LANKA) PLC",

    "C T HOLDINGS PLC": "C T HOLDINGS PLC",
    "CT HOLDINGS PLC": "C T HOLDINGS PLC",

    "THE COLOMBO FORT LAND & BUILDING PLC": "THE COLOMBO FORT LAND AND BUILDING PLC",
    "THE COLOMBO FORT LAND AND BUILDING PLC": "THE COLOMBO FORT LAND AND BUILDING PLC",
}

# =========================================================
# NORMALIZATION
# =========================================================

def clean_text(x):
    if x is None or pd.isna(x):
        return None
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x

def normalize_entity_name(name):
    name = clean_text(name)
    if not name:
        return None

    x = name.upper()

    # Remove honorifics
    x = re.sub(r"\b(MR|MRS|MS|MISS|DR|PROF)\.?\s+", "", x)

    # Normalize punctuation
    x = x.replace("&", " AND ")
    x = re.sub(r"[(),./]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    # Remove common suffix noise carefully
    x = re.sub(r"\bPRIVATE LIMITED\b", "PVT LTD", x)
    x = re.sub(r"\bPVT LIMITED\b", "PVT LTD", x)
    x = re.sub(r"\bLIMITED\b", "LTD", x)

    # Manual aliases
    x = MANUAL_ALIASES.get(x, x)

    return x

def make_node_id(entity_type, canonical_name, symbol=None):
    if symbol:
        return f"COMPANY::{symbol}"
    raw = f"{entity_type}::{canonical_name}"
    h = hashlib.md5(raw.encode("utf-8")).hexdigest()[:10]
    return f"{entity_type}::{h}"

def detect_entity_type(name, shareholder_type=None, is_listed=None, symbol=None):
    name_norm = normalize_entity_name(name) or ""

    if symbol:
        return "LISTED_COMPANY"

    if is_listed is True:
        return "LISTED_COMPANY"

    if shareholder_type:
        st = str(shareholder_type).lower()
        if st == "individual":
            return "PERSON"
        if st == "government":
            return "GOVERNMENT"
        if st == "institutional":
            return "UNLISTED_COMPANY"

    if any(k in name_norm for k in ["GOVERNMENT", "AUTHORITY", "MINISTRY", "TREASURY"]):
        return "GOVERNMENT"

    if any(k in name_norm for k in ["PLC", "LTD", "LIMITED", "PVT", "BANK", "FUND", "HOLDINGS", "COMPANY", "CORPORATION"]):
        return "UNLISTED_COMPANY"

    return "PERSON"

def safe_float(x):
    if x is None or pd.isna(x):
        return None
    try:
        return float(x)
    except:
        return None

# =========================================================
# LOAD COMPANY MASTER
# =========================================================

master = pd.read_csv(COMPANY_MASTER_CSV)
master.columns = master.columns.str.strip()

master["symbol_short"] = master["symbol_short"].astype(str).str.strip()
master["canonical_name"] = master["company_name"].apply(normalize_entity_name)
master["canonical_api_name"] = master["company_name_api"].apply(normalize_entity_name)

symbol_to_master = {}
name_to_symbol = {}

for _, r in master.iterrows():
    symbol = clean_text(r.get("symbol_short"))
    if not symbol or symbol == "nan":
        continue

    symbol_to_master[symbol] = r.to_dict()

    for n in [r.get("canonical_name"), r.get("canonical_api_name")]:
        if n:
            name_to_symbol[n] = symbol

def resolve_company_symbol(name=None, symbol=None):
    if symbol:
        s = str(symbol).replace(".N0000", "").strip()
        if s and s.lower() != "nan":
            return s

    n = normalize_entity_name(name)
    if n in name_to_symbol:
        return name_to_symbol[n]

    return None

# =========================================================
# LOAD OWNERSHIP JSON FILES
# =========================================================

def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        txt = f.read().strip()

    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        # Try to recover if file has extra text
        start = txt.find("[")
        end = txt.rfind("]")
        if start != -1 and end != -1:
            return json.loads(txt[start:end+1])
        raise

records = []
for path in glob.glob(OWNERSHIP_JSON_PATTERN):
    data = load_json_file(path)
    if isinstance(data, dict):
        data = [data]
    for item in data:
        item["_source_file"] = os.path.basename(path)
        records.append(item)

print("Ownership records loaded:", len(records))

# =========================================================
# NODE / EDGE BUILDERS
# =========================================================

nodes = {}
edges = []
aliases = []

def add_node(name, entity_type=None, symbol=None, extra=None):
    name = clean_text(name)
    if not name:
        return None

    symbol = resolve_company_symbol(name, symbol)
    canonical_name = normalize_entity_name(name)

    if symbol and symbol in symbol_to_master:
        m = symbol_to_master[symbol]
        display_name = clean_text(m.get("company_name_api")) or clean_text(m.get("company_name")) or name
        canonical_name = normalize_entity_name(display_name)
        entity_type = "LISTED_COMPANY"
        node_id = make_node_id(entity_type, canonical_name, symbol=symbol)

        node = {
            "node_id": node_id,
            "name": display_name,
            "canonical_name": canonical_name,
            "entity_type": entity_type,
            "cse_symbol": symbol,
            "company_id": m.get("company_id"),
            "sector": m.get("sector"),
            "industry_group": m.get("industry_group"),
            "board": m.get("board"),
            "status": m.get("status"),
        }
    else:
        entity_type = entity_type or detect_entity_type(name)
        node_id = make_node_id(entity_type, canonical_name)

        node = {
            "node_id": node_id,
            "name": name,
            "canonical_name": canonical_name,
            "entity_type": entity_type,
            "cse_symbol": None,
            "company_id": None,
            "sector": None,
            "industry_group": None,
            "board": None,
            "status": None,
        }

    if extra:
        for k, v in extra.items():
            if k not in node or node[k] in [None, "", "nan"]:
                node[k] = v

    if node_id not in nodes:
        nodes[node_id] = node

    aliases.append({
        "alias_name": name,
        "alias_normalized": normalize_entity_name(name),
        "node_id": node_id,
        "canonical_name": nodes[node_id]["canonical_name"],
        "entity_type": nodes[node_id]["entity_type"],
        "cse_symbol": nodes[node_id]["cse_symbol"],
    })

    return node_id

def add_edge(source_id, target_id, relationship_type, pct=None, share_count=None,
             source_url=None, confidence=None, report_company=None,
             report_symbol=None, data_source_date=None, evidence=None):

    if not source_id or not target_id:
        return

    edge_key_raw = f"{source_id}|{target_id}|{relationship_type}|{pct}|{data_source_date}"
    edge_id = hashlib.md5(edge_key_raw.encode("utf-8")).hexdigest()

    edges.append({
        "edge_id": edge_id,
        "source_node_id": source_id,
        "target_node_id": target_id,
        "relationship_type": relationship_type,
        "ownership_percentage": pct,
        "share_count": share_count,
        "source_url": source_url,
        "confidence": confidence,
        "report_company": report_company,
        "report_symbol": report_symbol,
        "data_source_date": data_source_date,
        "evidence": evidence,
    })

# =========================================================
# CONVERT JSON TO GRAPH
# =========================================================

for rec in records:
    company_name = clean_text(rec.get("company_name"))
    company_symbol = resolve_company_symbol(company_name, rec.get("cse_symbol"))
    confidence = safe_float(rec.get("confidence"))
    data_source_date = rec.get("data_source_date")
    source_file = rec.get("_source_file")

    if confidence is not None and confidence < MIN_CONFIDENCE:
        continue

    company_node = add_node(
        company_name,
        entity_type="LISTED_COMPANY",
        symbol=company_symbol,
        extra={"source_file": source_file}
    )

    # 1. Major shareholders: shareholder -> owns -> company
    for sh in rec.get("major_shareholders", []) or []:
        sh_name = sh.get("shareholder_name")
        sh_symbol = resolve_company_symbol(sh_name, None)
        sh_type = detect_entity_type(
            sh_name,
            shareholder_type=sh.get("shareholder_type"),
            symbol=sh_symbol
        )

        sh_node = add_node(
            sh_name,
            entity_type=sh_type,
            symbol=sh_symbol
        )

        add_edge(
            source_id=sh_node,
            target_id=company_node,
            relationship_type="OWNS",
            pct=safe_float(sh.get("ownership_percentage")),
            share_count=safe_float(sh.get("share_count")),
            source_url=sh.get("source"),
            confidence=confidence,
            report_company=company_name,
            report_symbol=company_symbol,
            data_source_date=data_source_date,
            evidence="major_shareholders"
        )

    # 2. Parent company: parent -> owns -> company
    parent = rec.get("parent_company") or {}
    if parent.get("name"):
        parent_symbol = resolve_company_symbol(parent.get("name"), None)
        parent_type = detect_entity_type(parent.get("name"), symbol=parent_symbol)

        parent_node = add_node(
            parent.get("name"),
            entity_type=parent_type,
            symbol=parent_symbol
        )

        add_edge(
            source_id=parent_node,
            target_id=company_node,
            relationship_type="PARENT_OF",
            pct=safe_float(parent.get("ownership_percentage")),
            source_url=parent.get("source"),
            confidence=confidence,
            report_company=company_name,
            report_symbol=company_symbol,
            data_source_date=data_source_date,
            evidence="parent_company"
        )

    # 3. Subsidiaries: company -> owns -> subsidiary
    for sub in rec.get("subsidiaries", []) or []:
        sub_name = sub.get("name")
        sub_symbol = resolve_company_symbol(sub_name, sub.get("cse_symbol"))
        sub_type = detect_entity_type(
            sub_name,
            is_listed=sub.get("is_listed"),
            symbol=sub_symbol
        )

        sub_node = add_node(
            sub_name,
            entity_type=sub_type,
            symbol=sub_symbol
        )

        add_edge(
            source_id=company_node,
            target_id=sub_node,
            relationship_type="SUBSIDIARY_OF",
            pct=safe_float(sub.get("ownership_percentage")),
            source_url=sub.get("source"),
            confidence=confidence,
            report_company=company_name,
            report_symbol=company_symbol,
            data_source_date=data_source_date,
            evidence="subsidiaries"
        )

    # 4. Associates: company -> associate_of -> associate
    for assoc in rec.get("associate_companies", []) or []:
        assoc_name = assoc.get("name")
        assoc_symbol = resolve_company_symbol(assoc_name, assoc.get("cse_symbol"))
        assoc_type = detect_entity_type(
            assoc_name,
            is_listed=assoc.get("is_listed"),
            symbol=assoc_symbol
        )

        assoc_node = add_node(
            assoc_name,
            entity_type=assoc_type,
            symbol=assoc_symbol
        )

        add_edge(
            source_id=company_node,
            target_id=assoc_node,
            relationship_type="ASSOCIATE_OF",
            pct=safe_float(assoc.get("ownership_percentage")),
            source_url=assoc.get("source"),
            confidence=confidence,
            report_company=company_name,
            report_symbol=company_symbol,
            data_source_date=data_source_date,
            evidence="associate_companies"
        )

    # 5. Investments in listed companies: company -> invests_in -> target
    for inv in rec.get("investments_in_listed_companies", []) or []:
        target_name = inv.get("target_company")
        target_symbol = resolve_company_symbol(target_name, inv.get("cse_symbol"))

        target_node = add_node(
            target_name,
            entity_type="LISTED_COMPANY" if target_symbol else "UNLISTED_COMPANY",
            symbol=target_symbol
        )

        add_edge(
            source_id=company_node,
            target_id=target_node,
            relationship_type="INVESTS_IN",
            pct=safe_float(inv.get("ownership_percentage")),
            source_url=inv.get("source"),
            confidence=confidence,
            report_company=company_name,
            report_symbol=company_symbol,
            data_source_date=data_source_date,
            evidence="investments_in_listed_companies"
        )

# =========================================================
# SAVE OUTPUTS
# =========================================================

nodes_df = pd.DataFrame(nodes.values()).drop_duplicates(subset=["node_id"])
edges_df = pd.DataFrame(edges).drop_duplicates(subset=["edge_id"])
aliases_df = pd.DataFrame(aliases).drop_duplicates()

# Manual review flags
edges_df["needs_review"] = False
edges_df.loc[edges_df["ownership_percentage"].isna(), "needs_review"] = True
edges_df.loc[edges_df["confidence"].fillna(0) < 0.9, "needs_review"] = True

nodes_df.to_csv(NODES_CSV, index=False, encoding="utf-8-sig")
edges_df.to_csv(EDGES_CSV, index=False, encoding="utf-8-sig")
aliases_df.to_csv(ALIASES_CSV, index=False, encoding="utf-8-sig")
edges_df[edges_df["needs_review"]].to_csv(REVIEW_CSV, index=False, encoding="utf-8-sig")

print("Saved:", NODES_CSV, len(nodes_df))
print("Saved:", EDGES_CSV, len(edges_df))
print("Saved:", ALIASES_CSV, len(aliases_df))
print("Saved:", REVIEW_CSV, edges_df["needs_review"].sum())

print("\nNode type counts:")
print(nodes_df["entity_type"].value_counts())

print("\nEdge type counts:")
print(edges_df["relationship_type"].value_counts())

Ownership records loaded: 303
Saved: cse_graph/nodes.csv 476
Saved: cse_graph/edges.csv 590
Saved: cse_graph/entity_aliases.csv 592
Saved: cse_graph/edges_manual_review.csv 253

Node type counts:
entity_type
LISTED_COMPANY      291
UNLISTED_COMPANY    135
PERSON               42
GOVERNMENT            8
Name: count, dtype: int64

Edge type counts:
relationship_type
OWNS             259
PARENT_OF        208
SUBSIDIARY_OF    115
ASSOCIATE_OF       7
INVESTS_IN         1
Name: count, dtype: int64


In [3]:
! pip install pyvis networkx pandas -q

In [4]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

nodes = pd.read_csv("cse_graph/nodes.csv")
edges = pd.read_csv("cse_graph/edges.csv")

# optional: keep cleaner edges first
edges = edges[
    (edges["confidence"] >= 0.85) &
    (edges["ownership_percentage"].notna())
].copy()

G = nx.DiGraph()

for _, row in nodes.iterrows():
    G.add_node(
        row["node_id"],
        label=row["name"],
        title=f"""
        Type: {row['entity_type']}<br>
        Symbol: {row.get('cse_symbol', '')}<br>
        Sector: {row.get('sector', '')}
        """,
        group=row["entity_type"]
    )

for _, row in edges.iterrows():
    G.add_edge(
        row["source_node_id"],
        row["target_node_id"],
        title=f"""
        Relationship: {row['relationship_type']}<br>
        Ownership: {row['ownership_percentage']}%<br>
        Confidence: {row['confidence']}<br>
        Source: {row['source_url']}
        """,
        label=f"{row['ownership_percentage']}%"
    )

net = Network(
    height="850px",
    width="100%",
    directed=True,
    notebook=False
)

net.from_nx(G)
net.show_buttons(filter_=["physics"])
net.write_html("cse_ownership_graph.html")

print("Saved: cse_ownership_graph.html")

Saved: cse_ownership_graph.html


### Load into neo4j cloud

In [6]:
! pip install neo4j pandas -q

In [8]:
import pandas as pd
from neo4j import GraphDatabase

URI = "neo4j+s://044142d8.databases.neo4j.io"
USERNAME = "044142d8"
PASSWORD = "YgokiTeLnPweC_mZj3JsP8JTlKM-23bUc7wCP6oCX1M"

nodes = pd.read_csv("cse_graph/nodes.csv").fillna("")
edges = pd.read_csv("cse_graph/edges.csv").fillna("")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

def run_query(query, params=None):
    with driver.session() as session:
        session.run(query, params or {})

# 1. Constraints
run_query("""
CREATE CONSTRAINT entity_node_id IF NOT EXISTS
FOR (e:Entity)
REQUIRE e.node_id IS UNIQUE
""")

# 2. Clear old graph, optional
run_query("MATCH (n) DETACH DELETE n")

# 3. Load nodes
def create_nodes(tx, batch):
    tx.run("""
    UNWIND $rows AS row
    MERGE (e:Entity {node_id: row.node_id})
    SET e.name = row.name,
        e.canonical_name = row.canonical_name,
        e.entity_type = row.entity_type,
        e.cse_symbol = CASE WHEN row.cse_symbol = "" THEN null ELSE row.cse_symbol END,
        e.company_id = CASE WHEN row.company_id = "" THEN null ELSE row.company_id END,
        e.sector = CASE WHEN row.sector = "" THEN null ELSE row.sector END,
        e.industry_group = CASE WHEN row.industry_group = "" THEN null ELSE row.industry_group END,
        e.board = CASE WHEN row.board = "" THEN null ELSE row.board END,
        e.status = CASE WHEN row.status = "" THEN null ELSE row.status END
    """, rows=batch)

def create_edges(tx, batch):
    tx.run("""
    UNWIND $rows AS row
    MATCH (source:Entity {node_id: row.source_node_id})
    MATCH (target:Entity {node_id: row.target_node_id})
    MERGE (source)-[r:RELATED_TO {edge_id: row.edge_id}]->(target)
    SET r.relationship_type = row.relationship_type,
        r.ownership_percentage =
            CASE WHEN row.ownership_percentage = "" THEN null ELSE toFloat(row.ownership_percentage) END,
        r.share_count =
            CASE WHEN row.share_count = "" THEN null ELSE toFloat(row.share_count) END,
        r.source_url = CASE WHEN row.source_url = "" THEN null ELSE row.source_url END,
        r.confidence =
            CASE WHEN row.confidence = "" THEN null ELSE toFloat(row.confidence) END,
        r.report_company = row.report_company,
        r.report_symbol = row.report_symbol,
        r.data_source_date = row.data_source_date,
        r.evidence = row.evidence
    """, rows=batch)

def batches(df, size=500):
    records = df.to_dict("records")
    for i in range(0, len(records), size):
        yield records[i:i+size]

with driver.session() as session:
    for batch in batches(nodes, 500):
        session.execute_write(create_nodes, batch)

    for batch in batches(edges, 500):
        session.execute_write(create_edges, batch)

# 4. Add labels
run_query("MATCH (e:Entity) WHERE e.entity_type = 'LISTED_COMPANY' SET e:ListedCompany")
run_query("MATCH (e:Entity) WHERE e.entity_type = 'UNLISTED_COMPANY' SET e:UnlistedCompany")
run_query("MATCH (e:Entity) WHERE e.entity_type = 'PERSON' SET e:Person")
run_query("MATCH (e:Entity) WHERE e.entity_type = 'GOVERNMENT' SET e:Government")

driver.close()

print("Neo4j Aura import completed.")

Neo4j Aura import completed.


In [11]:
from neo4j import GraphDatabase
import pandas as pd



driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

# ==============================
# GENERIC QUERY FUNCTION
# ==============================
df = run_query("""
MATCH (c:ListedCompany)
RETURN c.name, c.cse_symbol, c.sector
ORDER BY c.name
""")

print(df.head())

# ==============================
# CLOSE CONNECTION
# ==============================
def close():
    driver.close()

                   c.name c.cse_symbol                     c.sector
0   ABANS ELECTRICALS PLC         ABAN  Consumer Durables & Apparel
1       ABANS FINANCE PLC          NaN                          NaN
2               ABANS PLC          NaN                          NaN
3  ACCESS ENGINEERING PLC          AEL                Capital Goods
4          ACL CABLES PLC          ACL                Capital Goods


## Run Streamlit

In [14]:
! pip install streamlit neo4j pandas pyvis

  Using cached streamlit-1.57.0-py3-none-any.whl.metadata (9.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 1.2 MB/s  0:00:07 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 kB 1.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 1.2 MB/s  0:00:09 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 1.5 MB/s  0:00:22m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [streamlit]19 [streamlit]


In [16]:
import streamlit.components.v1 as components